# RO-075 The autonomy stack: every number in the Note

The room, the person and the local planner's drive are RO-058's (`images/bandkit.py` and `images/trajkit.py`, copied;
the drive is computed once and cached). The sense-plan-act loop, the Braitenberg vehicles and the subsumption robot
live in `images/stackkit.py`. Nothing is random, so there is no seed to set.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import bandkit as bk
import stackkit as sk
from trajkit import R_BODY, A_MAX, V_WHEEL, L_AXLE
r2, r3, r4 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))
R = bk.results(); L, F = R['loop'], R['free']

## 1. The person at 2.4 s (Overview)

In [2]:
c = L[24]
x, y, th = c['x'][:3]
gap = float(sk.person_gap(c['x'][:2])[0])
bearing = math.degrees(math.atan2(bk.PERSON[1] - y, bk.PERSON[0] - x) - th)
print('t', c['t'], 's: robot at', np.round(c['x'][:2], 3), 'heading', r2(math.degrees(th)), 'deg, speed', r3(c['v']), 'm/s')
print('gap from the robot disk to the person:', r3(gap), 'm; the person is', r2(bearing), 'deg from straight ahead')
print('with the local planner (RO-058): door at', L[-1]['t'], 's; clear room', F[-1]['t'], 's; closest to the person',
      r3(bk.gap(*np.array([o['x'][:2] for o in L]).T, bk.PERSON).min()), 'm')

t 2.4 s: robot at [2.495 1.929] heading 69.15 deg, speed 0.595 m/s
gap from the robot disk to the person: 0.465 m; the person is -5.41 deg from straight ahead
with the local planner (RO-058): door at 8.0 s; clear room 6.7 s; closest to the person 0.14 m


## 2. Sense-plan-act: how often must the loop look? (Section 2, Figures 1 and 3)

In [3]:
v, a = 0.595, A_MAX
stop = v ** 2 / (2 * a)
print('stopping distance from', v, 'm/s at', a, 'm/s^2:', r3(stop), 'm')
g = 0.465
print('longest loop period that still stops: (', g, '-', r3(stop), ') /', v, '=', r3((g - r3(stop)) / v), 's')
for T in (1.0, 0.5, 0.2, 0.1, 0.05):
    d = v * T   # the sum is printed from the rounded parts, as the Note's table adds them
    print('period', T, 's: travel before the look', r3(d), 'm; plus braking', r3(r3(d) + r3(stop)), 'm',
          'stops' if d + stop < g else 'touches')
for T in (1.0, 0.1):
    s = sk.spa(T)
    k = np.argmax(s['gap'] < 0) if (s['gap'] < 0).any() else None
    print('loop every', T, 's: looks at', r2(s['t_look']), 's, gap then', r3(s['gap_look']), 'm at', r3(s['v_look']), 'm/s;',
          'touches at %.2f s at %.3f m/s' % (s['t'][k], s['v'][k]) if k is not None else 'stops %.3f m short at %.2f s' % (s['gap'].min(), s['t'][-1]))
    print('  braking from the look at', r3(s['v_look']), 'm/s needs', r3(s['v_look'] ** 2 / (2 * a)), 'm')
print('distance covered from 2.4 s to 3.0 s on the old plan:', r3(np.hypot(*(F[30]['x'][:2] - F[24]['x'][:2]))), 'm (straight line)')

stopping distance from 0.595 m/s at 0.5 m/s^2: 0.354 m
longest loop period that still stops: ( 0.465 - 0.354 ) / 0.595 = 0.187 s
period 1.0 s: travel before the look 0.595 m; plus braking 0.949 m touches
period 0.5 s: travel before the look 0.297 m; plus braking 0.651 m touches
period 0.2 s: travel before the look 0.119 m; plus braking 0.473 m touches
period 0.1 s: travel before the look 0.059 m; plus braking 0.413 m stops
period 0.05 s: travel before the look 0.03 m; plus braking 0.384 m stops
loop every 1.0 s: looks at 3.0 s, gap then 0.113 m at 0.591 m/s; touches at 3.22 s at 0.481 m/s
  braking from the look at 0.591 m/s needs 0.35 m
loop every 0.1 s: looks at 2.4 s, gap then 0.465 m at 0.595 m/s; stops 0.112 m short at 3.59 s
  braking from the look at 0.595 m/s needs 0.354 m
distance covered from 2.4 s to 3.0 s on the old plan: 0.356 m (straight line)


## 3. Braitenberg vehicles (Section 3, Figures 4 to 6)

In [4]:
x, y, th = 2.0, 1.0, math.radians(30)
for side, name in ((1, 'left'), (-1, 'right')):
    a_s = th + side * sk.SENSOR_ANGLE
    sx, sy = x + R_BODY * math.cos(a_s), y + R_BODY * math.sin(a_s)
    dx, dy = sk.LAMP[0] - sx, sk.LAMP[1] - sy
    d = math.hypot(dx, dy)
    ang = math.degrees(math.atan2(dy, dx)) - math.degrees(a_s)
    facing = (1 + math.cos(math.radians(ang))) / 2
    I = 1 / (1 + (d / sk.D0) ** 2)
    print(name, 'sensor faces', r2(math.degrees(a_s)), 'deg; lamp', r3(d), 'm away,', r2(ang), 'deg off its axis;',
          'brightness', r4(I), 'facing', r4(facing), 'reading', r4(I * facing))
sL, sR = sk.light((x, y, th))
vL, vR = sk.braitenberg_wheels(sL, sR, '3a')
print('readings', r4(sL), r4(sR), '; wheels left', r4(vL), 'right', r4(vR), '; v', r4((vL + vR) / 2), 'omega', r4((vR - vL) / L_AXLE))

left sensor faces 75.0 deg; lamp 2.731 m away, -19.21 deg off its axis; brightness 0.1182 facing 0.9722 reading 0.1149
right sensor faces -15.0 deg; lamp 2.902 m away, 77.09 deg off its axis; brightness 0.1061 facing 0.6117 reading 0.0649
readings 0.1149 0.0649 ; wheels left 0.4621 right 0.5221 ; v 0.4921 omega 0.3002


In [5]:
for w in ('2a', '2b', '3a', '3b'):
    rec = sk.run_open(w, t_end=15)
    xs = np.array([o['x'] for o in rec]); d = np.hypot(xs[:, 0], xs[:, 1])
    print(w, ': start', r2(d[0]), 'm from the lamp; closest', r2(d.min()), 'm at', rec[int(np.argmin(d))]['t'], 's; end',
          r2(d[-1]), 'm at', rec[-1]['t'], 's, speed', r3(xs[-1, 3]), 'm/s; top speed', r3(xs[:, 3].max()),
          '; at 8 s:', r2(d[min(800, len(d) - 1)]), 'm, speed', r3(xs[min(800, len(d) - 1), 3]), 'm/s')

2a : start 1.62 m from the lamp; closest 1.49 m at 0.75 s; end 1.93 m at 15.0 s, speed 0.026 m/s; top speed 0.309 ; at 8 s: 1.74 m, speed 0.029 m/s
2b : start 1.62 m from the lamp; closest 0.25 m at 2.85 s; end 0.25 m at 2.85 s, speed 0.364 m/s; top speed 0.6 ; at 8 s: 0.25 m, speed 0.364 m/s
3a : start 1.62 m from the lamp; closest 0.87 m at 15.0 s; end 0.87 m at 15.0 s, speed 0.0 m/s; top speed 0.295 ; at 8 s: 0.89 m, speed 0.01 m/s
3b : start 1.62 m from the lamp; closest 1.48 m at 0.74 s; end 9.61 m at 15.0 s, speed 0.598 m/s; top speed 0.598 ; at 8 s: 5.43 m, speed 0.595 m/s


In [6]:
rec = sk.run_braitenberg()
xs = np.array([o['x'] for o in rec])
print('vehicle 3a in the room: touches the table at', rec[-1]['t'], 's at', np.round(xs[-1, :2], 3), 'speed', r4(xs[-1, 3]),
      'm/s;', r3(np.hypot(*(xs[-1, :2] - sk.LAMP))), 'm from the lamp')

vehicle 3a in the room: touches the table at 9.84 s at [3.163 2.685] speed 0.0248 m/s; 0.925 m from the lamp


## 4. Subsumption (Section 4, Figure 8)

In [7]:
rec = sk.run_subsumption(t_end=20)
xs = np.array([o['x'] for o in rec])
h = [o for o in rec if o['who'] == 'collide: halt']
print('first halt at', h[0]['t'], 's, gap to the person', r3(sk.person_gap(h[0]['x'][:2])[0]), 'm, speed', r3(h[0]['x'][3]))
print('stopped (below 0.01 m/s) at', [o['t'] for o in rec if o['t'] > h[0]['t'] and abs(o['x'][3]) < 0.01][0], 's')
print('at 20 s: gap to the person', r3(sk.person_gap(xs[-1, :2])[0]), 'm;', r3(np.hypot(*(xs[-1, :2] - sk.LAMP))), 'm from the lamp;',
      'touched anything:', any(o['hit'] for o in rec))
print('steps driven by each module:', {k: sum(o['who'] == k for o in rec) for k in ('go to the light', 'collide: halt')})

first halt at 3.2 s, gap to the person 0.411 m, speed 0.363
stopped (below 0.01 m/s) at 3.91 s
at 20 s: gap to the person 0.283 m; 1.748 m from the lamp; touched anything: False
steps driven by each module: {'go to the light': 320, 'collide: halt': 1681}


In [8]:
# the same run with the two numbers of the push changed, one grid of nine settings
res = []
for fd in (0.4, 0.5, 0.6):
    for kt in (1.0, 2.0, 3.0):
        sk.FORCE_DIST, sk.K_TURN = fd, kt
        rr = sk.run_subsumption(t_end=20)
        xx = np.array([o['x'] for o in rr])
        res.append((fd, kt, float(np.hypot(xx[:, 0] - sk.LAMP[0], xx[:, 1] - sk.LAMP[1]).min()),
                    float(bk.gap(xx[:, 0], xx[:, 1], bk.PERSON).min()), any(o['hit'] for o in rr)))
sk.FORCE_DIST, sk.K_TURN = 0.5, 2.0
for r in res: print(r)
print('closest any setting came to the lamp:', r3(min(r[2] for r in res)), 'm; closest to anything:', r3(min(r[3] for r in res)),
      'm; any touch:', any(r[4] for r in res))

(0.4, 1.0, 1.7231160361566162, 0.2835466058102982, False)
(0.4, 2.0, 1.7289967243922768, 0.28024078042864975, False)
(0.4, 3.0, 1.6211530789814184, 0.22504272034008688, False)
(0.5, 1.0, 1.7391630244950644, 0.27814795323018826, False)
(0.5, 2.0, 1.7481407333114625, 0.2826697091870527, False)
(0.5, 3.0, 1.6886244479562726, 0.22442979450508416, False)
(0.6, 1.0, 1.7393443044452916, 0.26540002751204494, False)
(0.6, 2.0, 1.1459386374662024, 0.16599354816901757, False)
(0.6, 3.0, 1.7453762547367602, 0.13887551625751715, False)
closest any setting came to the lamp: 1.146 m; closest to anything: 0.139 m; any touch: False


## 5. The layered stack (Section 5, Figure 9)

In [9]:
for k in (23, 24, 25):
    o = L[k]
    print(o['t'], 's: speed', r3(o['v']), '; local plan asks v', r3(o['cmd'][0]), 'omega', r3(o['cmd'][1]), '; poses', len(o['P']))
# the wheel speeds the robot actually drove in the cycle at 2.4 s (the drive of bandkit.run_loop, replayed)
for k in (24, 25):
    o = L[k]; P, DT = o['P'], o['DT']; v = o['v']
    vv, ww, _, _, _ = bk.band_terms(P, DT, v, o['final'])
    Tb = np.concatenate([[0], np.cumsum(DT)]); hh = bk.CYCLE / 20
    for j in range(20):
        tm = (j + 0.5) * hh; i = min(np.searchsorted(Tb, tm, side='right') - 1, len(DT) - 1)
        v_new = float(np.clip(v + np.clip(vv[i] - v, -A_MAX * hh, A_MAX * hh), 0, V_WHEEL))
        lim = (V_WHEEL - v_new) / (L_AXLE / 2); w_new = float(np.clip(ww[i], -lim, lim))
        if j in (0, 10):
            print('  t', r3(o['t'] + j * hh), 'wheels right', r3(v_new + w_new * L_AXLE / 2), 'left', r3(v_new - w_new * L_AXLE / 2))
        v = v_new
Pw, Dw = L[24]['warm']
def closest(P):
    seg = np.concatenate([np.linspace(P[k, :2], P[k + 1, :2], 20) for k in range(len(P) - 1)])
    return float(bk.gap(seg[:, 0], seg[:, 1], bk.PERSON).min())
print('the band at 2.4 s asks', r3(L[24]['cmd'][0]), 'm/s and', r3(L[24]['cmd'][1]), 'rad/s: right wheel would need',
      r3(L[24]['cmd'][0] + L[24]['cmd'][1] * L_AXLE / 2), 'm/s, above the 0.6 m/s limit')
print('band at 2.4 s: warm start', r3(closest(Pw)), 'm from the person (overlap), solved', r3(closest(L[24]['P'])), 'm clear')
for T in (1.0, 0.1, 0.05, 0.01):
    print('at 0.6 m/s the robot covers', r3(0.6 * T), 'm in', T, 's')

2.3 s: speed 0.583 ; local plan asks v 0.595 omega 0.343 ; poses 9
2.4 s: speed 0.595 ; local plan asks v 0.439 omega 2.066 ; poses 10
2.5 s: speed 0.545 ; local plan asks v 0.437 omega 2.123 ; poses 10
  t 2.4 wheels right 0.6 left 0.584
  t 2.45 wheels right 0.6 left 0.534
  t 2.5 wheels right 0.6 left 0.484
  t 2.55 wheels right 0.6 left 0.434
the band at 2.4 s asks 0.439 m/s and 2.066 rad/s: right wheel would need 0.646 m/s, above the 0.6 m/s limit
band at 2.4 s: warm start -0.218 m from the person (overlap), solved 0.146 m clear
at 0.6 m/s the robot covers 0.6 m in 1.0 s
at 0.6 m/s the robot covers 0.06 m in 0.1 s
at 0.6 m/s the robot covers 0.03 m in 0.05 s
at 0.6 m/s the robot covers 0.006 m in 0.01 s


## 6. Sensors (Section 6, Figure 11)

In [10]:
react, brake = 0.6 * 0.1, 0.6 ** 2 / (2 * 0.5)
print('range needed from the rim:', r3(react), '+', r3(brake), '=', r3(react + brake), 'm; from the centre:', r3(react + brake + R_BODY), 'm')
steps = 240 / 0.352
print('Hokuyo URG-04LX-UG01: 240 deg in 0.352 deg steps =', round(steps), 'readings per scan; blind sector', 360 - 240, 'deg')
print('scan data:', round(steps), 'x 4 bytes x 10 per second =', r3(round(steps) * 4 * 10 / 1000), 'kB/s')
pts = 1280 * 720
print('RealSense D435 depth at 1280 x 720 and 30 per second:', pts, 'points per frame,', pts * 30, 'per second;',
      round(pts * 30 / (round(steps) * 10)), 'times the scanner')
print('Nav2 local costmap: 3 m / 0.05 m =', round(3 / 0.05), 'cells per beam;', round(3 / 0.05) * round(steps), 'cell visits per scan at most')
print('sonar, echo from 5 m at about 300 m/s (Correll 5.4):', r3(10 / 300), 's per reading,', round(1 / (10 / 300)), 'readings per second')
print('laser readings per second:', round(steps) * 10, '; camera points per second / laser readings per second =', round(pts * 30 / (round(steps) * 10)))
print('backing up 0.30 m at 0.15 m/s (Nav2 BackUp) takes', 0.30 / 0.15, 's')
b, rr = sk.ranges(tuple(L[24]['x'][:3]), bk.PERSON, n=360, rmax=5.6, origin=0.0)
k = int(np.argmin(np.where(np.abs(np.degrees(b)) < 30, rr, 99)))
print('scan from the centre at 2.4 s: nearest reading within 30 deg of ahead', r3(rr[k]), 'm at', r2(np.degrees(b[k])), 'deg')

range needed from the rim: 0.06 + 0.36 = 0.42 m; from the centre: 0.67 m
Hokuyo URG-04LX-UG01: 240 deg in 0.352 deg steps = 682 readings per scan; blind sector 120 deg
scan data: 682 x 4 bytes x 10 per second = 27.28 kB/s
RealSense D435 depth at 1280 x 720 and 30 per second: 921600 points per frame, 27648000 per second; 4054 times the scanner
Nav2 local costmap: 3 m / 0.05 m = 60 cells per beam; 40920 cell visits per scan at most
sonar, echo from 5 m at about 300 m/s (Correll 5.4): 0.033 s per reading, 30 readings per second
laser readings per second: 6820 ; camera points per second / laser readings per second = 4054
backing up 0.30 m at 0.15 m/s (Nav2 BackUp) takes 2.0 s
scan from the centre at 2.4 s: nearest reading within 30 deg of ahead 0.715 m at -6.0 deg
